## Preparando os dados via API

In [0]:
!pip install ccxt

In [0]:
%skip
symbols = list(binance.load_markets().keys())
df_symbols = spark.createDataFrame([(s,) for s in symbols], ["symbol"])
df_symbols.display()

In [0]:
import ccxt
import time
from typing import List, Dict, Tuple
import pyspark.sql.types as t
import pyspark.sql.functions as f
from datetime import datetime, timedelta

# Configuração de pares a coletar (top liquidity coins)
PAIRS = [
    'BTC/USDT',
    'ETH/USDT', 
    'SOL/USDT',
    'NEAR/USDT',
    'BNB/USDT',
    'MATIC/USDT',
    'AVAX/USDT',
    'LINK/USDT'
]

# Múltiplos timeframes para features multi-escala
TIMEFRAMES = ['15m', '1h', '4h']

# Período: 3 anos de histórico
START_DATE = '2021-04-01T00:00:00Z'  # ~3 anos atrás
MAX_CANDLES_TARGET = 50000  # limite seguro

# Configuração da API
binance = ccxt.binanceus({
    'enableRateLimit': True,
    'timeout': 30000,  # 30s timeout
    'options': {'defaultType': 'spot'}
})

print(f"📊 Configuração de Coleta:")
print(f"  • Pares: {len(PAIRS)}")
print(f"  • Timeframes: {TIMEFRAMES}")
print(f"  • Início: {START_DATE}")
print(f"  • Exchange: {binance}")

In [0]:
def fetch_ohlcv_robust(
    exchange: ccxt.Exchange,
    symbol: str,
    timeframe: str,
    start_date: str,
    max_candles: int = 50000,
    batch_size: int = 1000,
    max_retries: int = 5
) -> List[List]:
    """
    Coleta dados OHLCV com retry logic robusto e tratamento de erros.
    
    Args:
        exchange: Instância ccxt da exchange
        symbol: Par de trading (ex: 'BTC/USDT')
        timeframe: Intervalo de tempo ('15m', '1h', '4h')
        start_date: Data inicial ISO 8601 format
        max_candles: Número máximo de candles a coletar
        batch_size: Tamanho do lote por requisição
        max_retries: Número máximo de tentativas por requisição
    
    Returns:
        Lista de candles OHLCV
    """
    all_candles = []
    since = exchange.parse8601(start_date)
    
    print(f"  🔄 Coletando {symbol} [{timeframe}]...", end="")
    
    while len(all_candles) < max_candles:
        retry_count = 0
        success = False
        
        while retry_count < max_retries and not success:
            try:
                # Tenta buscar o lote
                new_candles = exchange.fetch_ohlcv(
                    symbol, 
                    timeframe=timeframe, 
                    since=since, 
                    limit=batch_size
                )
                
                # Se não retornou nada, chegamos ao fim
                if not new_candles:
                    print(f" ✅ Fim do histórico ({len(all_candles)} candles)")
                    return all_candles
                
                # Adiciona novos candles
                all_candles.extend(new_candles)
                
                # Atualiza timestamp para próxima iteração
                since = new_candles[-1][0] + 1
                
                # Pequena pausa para respeitar rate limit
                time.sleep(exchange.rateLimit / 1000)
                
                success = True
                
                # Feedback de progresso a cada 5000 candles
                if len(all_candles) % 5000 == 0:
                    print(f" {len(all_candles)}...", end="")
                
            except ccxt.NetworkError as e:
                retry_count += 1
                wait_time = 2 ** retry_count  # Exponential backoff
                print(f" ⚠️ Network error (retry {retry_count}/{max_retries})", end="")
                time.sleep(wait_time)
                
            except ccxt.ExchangeError as e:
                retry_count += 1
                wait_time = 2 ** retry_count
                print(f" ⚠️ Exchange error (retry {retry_count}/{max_retries})", end="")
                time.sleep(wait_time)
                
            except Exception as e:
                print(f" ❌ Erro inesperado: {str(e)[:100]}")
                return all_candles  # Retorna o que conseguimos coletar
        
        # Se esgotou todas as tentativas
        if not success:
            print(f" ❌ Falhou após {max_retries} tentativas")
            return all_candles
    
    print(f" ✅ Completo ({len(all_candles)} candles)")
    return all_candles

print("✅ Função de coleta definida")

In [0]:
from datetime import datetime

# Dicionário para armazenar todos os dados coletados
# Estrutura: collected_data[timeframe][symbol] = list_of_candles
collected_data = {tf: {} for tf in TIMEFRAMES}

start_time = datetime.now()
print(f"\n🚀 Iniciando coleta em larga escala...")
print(f"   Hora de início: {start_time.strftime('%H:%M:%S')}\n")

total_pairs = len(PAIRS) * len(TIMEFRAMES)
current = 0

for timeframe in TIMEFRAMES:
    print(f"\n📅 Timeframe: {timeframe}")
    print("=" * 60)
    
    for symbol in PAIRS:
        current += 1
        print(f"[{current}/{total_pairs}] ", end="")
        
        try:
            candles = fetch_ohlcv_robust(
                exchange=binance,
                symbol=symbol,
                timeframe=timeframe,
                start_date=START_DATE,
                max_candles=MAX_CANDLES_TARGET
            )
            
            collected_data[timeframe][symbol] = candles
            
        except Exception as e:
            print(f"  ❌ ERRO crítico em {symbol}: {str(e)[:100]}")
            collected_data[timeframe][symbol] = []  # Lista vazia para não quebrar pipeline

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\n{'='*60}")
print(f"✅ Coleta concluída em {duration:.1f}s ({duration/60:.1f} min)")
print(f"\n📊 Resumo da coleta:")

total_candles = 0
for tf in TIMEFRAMES:
    tf_total = sum(len(candles) for candles in collected_data[tf].values())
    total_candles += tf_total
    print(f"  • {tf}: {tf_total:,} candles")

print(f"\n📊 Total geral: {total_candles:,} candles")

In [0]:
# Schema para OHLCV
schema = t.StructType([
    t.StructField("timestamp", t.LongType(), True),
    t.StructField("open", t.DoubleType(), True),
    t.StructField("high", t.DoubleType(), True),
    t.StructField("low", t.DoubleType(), True),
    t.StructField("close", t.DoubleType(), True),
    t.StructField("volume", t.DoubleType(), True)
])

print("\n📦 Convertendo para DataFrames Spark...\n")

# Dicionário para armazenar DataFrames
# Estrutura: spark_dfs[timeframe][symbol] = DataFrame
spark_dfs = {tf: {} for tf in TIMEFRAMES}

for timeframe in TIMEFRAMES:
    print(f"📅 {timeframe}:")
    
    for symbol in PAIRS:
        candles = collected_data[timeframe][symbol]
        
        if len(candles) > 0:
            # Cria DataFrame
            df = spark.createDataFrame(candles, schema)
            
            # Converte timestamp de milissegundos para timestamp SQL
            df = df.withColumn(
                "timestamp", 
                (f.col("timestamp") / 1000).cast("timestamp")
            )
            
            # Converte para fuso horário de São Paulo
            df = df.withColumn(
                "timestamp", 
                f.from_utc_timestamp("timestamp", "America/Sao_Paulo")
            )
            
            # Adiciona metadados
            df = df.withColumn("symbol", f.lit(symbol))
            df = df.withColumn("timeframe", f.lit(timeframe))
            
            # Ordena por timestamp
            df = df.orderBy("timestamp")
            
            spark_dfs[timeframe][symbol] = df
            
            print(f"  ✅ {symbol}: {df.count():,} registros")
        else:
            print(f"  ⚠️ {symbol}: sem dados")

print("\n✅ Conversão concluída")

## Consolidação e Validação dos Dados

Agora vamos:
1. Consolidar todos os pares em tabelas únicas por timeframe
2. Validar a qualidade dos dados (nulos, duplicatas, gaps)
3. Salvar em Delta Lake para uso no modelo

In [0]:
print("\n📦 Consolidando dados por timeframe...\n")

# Dicionário para armazenar as tabelas consolidadas
consolidated_tables = {}

for timeframe in TIMEFRAMES:
    print(f"📅 Consolidando {timeframe}:")
    
    # Lista de DataFrames para unir
    dfs_to_union = []
    
    for symbol in PAIRS:
        if symbol in spark_dfs[timeframe]:
            df = spark_dfs[timeframe][symbol]
            
            # Adiciona apenas se tiver dados
            if df.count() > 0:
                dfs_to_union.append(df)
    
    # Faz union de todos os DataFrames
    if len(dfs_to_union) > 0:
        consolidated_df = dfs_to_union[0]
        
        for df in dfs_to_union[1:]:
            consolidated_df = consolidated_df.union(df)
        
        # Ordena por symbol e timestamp
        consolidated_df = consolidated_df.orderBy("symbol", "timestamp")
        
        # Remove duplicatas (caso existam)
        consolidated_df = consolidated_df.dropDuplicates(["symbol", "timestamp"])
        
        consolidated_tables[timeframe] = consolidated_df
        
        # Estatísticas
        total_rows = consolidated_df.count()
        symbol_counts = consolidated_df.groupBy("symbol").count().orderBy("symbol")
        
        print(f"  ✅ Total: {total_rows:,} registros")
        print(f"  • Distribuição por par:")
        for row in symbol_counts.collect():
            print(f"     - {row['symbol']}: {row['count']:,}")
    else:
        print(f"  ⚠️ Nenhum dado para consolidar")
    
    print()

print("✅ Consolidação concluída")

In [0]:
print("\n🔍 Validando qualidade dos dados...\n")

for timeframe in TIMEFRAMES:
    if timeframe not in consolidated_tables:
        continue
    
    df = consolidated_tables[timeframe]
    
    print(f"📅 {timeframe}:")
    print("=" * 60)
    
    # 1. Verifica nulos
    null_counts = df.select(
        [
            f.sum(f.when(f.col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in ["open", "high", "low", "close", "volume"]
        ]
    ).collect()[0].asDict()
    
    has_nulls = any(count > 0 for count in null_counts.values())
    
    if has_nulls:
        print(f"  ⚠️ Nulos encontrados:")
        for col, count in null_counts.items():
            if count > 0:
                print(f"     - {col}: {count}")
    else:
        print(f"  ✅ Nenhum valor nulo")
    
    # 2. Verifica range de datas
    date_stats = df.agg(
        f.min("timestamp").alias("min_date"),
        f.max("timestamp").alias("max_date")
    ).collect()[0]
    
    print(f"  • Período: {date_stats['min_date']} a {date_stats['max_date']}")
    
    # 3. Verifica valores negativos ou zeros
    invalid_prices = df.filter(
        (f.col("open") <= 0) | 
        (f.col("high") <= 0) | 
        (f.col("low") <= 0) | 
        (f.col("close") <= 0)
    ).count()
    
    if invalid_prices > 0:
        print(f"  ⚠️ Preços inválidos (≤ 0): {invalid_prices}")
    else:
        print(f"  ✅ Todos os preços válidos (> 0)")
    
    # 4. Verifica consistência OHLC (high >= low, etc)
    inconsistent_ohlc = df.filter(
        (f.col("high") < f.col("low")) |
        (f.col("high") < f.col("open")) |
        (f.col("high") < f.col("close")) |
        (f.col("low") > f.col("open")) |
        (f.col("low") > f.col("close"))
    ).count()
    
    if inconsistent_ohlc > 0:
        print(f"  ⚠️ Candles inconsistentes: {inconsistent_ohlc}")
    else:
        print(f"  ✅ OHLC consistente")
    
    print()

print("✅ Validação concluída")

In [0]:
# Mostra uma amostra de dados para cada par (timeframe 1h)
if '1h' in consolidated_tables:
    print("\n📊 Amostra dos dados (timeframe 1h):\n")
    
    df_1h = consolidated_tables['1h']
    
    # Pega últimos 5 registros de cada par
    from pyspark.sql.window import Window
    
    w = Window.partitionBy("symbol").orderBy(f.col("timestamp").desc())
    
    sample_df = (
        df_1h
        .withColumn("row_num", f.row_number().over(w))
        .filter(f.col("row_num") <= 5)
        .drop("row_num")
        .orderBy("symbol", f.col("timestamp").desc())
    )
    
    display(sample_df)

In [0]:
print("\n💾 Salvando tabelas em Delta Lake...\n")

for timeframe in TIMEFRAMES:
    if timeframe not in consolidated_tables:
        continue
    
    df = consolidated_tables[timeframe]
    table_name = f"default.crypto_ohlcv_{timeframe.replace('m', 'min').replace('h', 'hour')}"
    
    print(f"  • Salvando {table_name}...", end=" ")
    
    try:
        (
            df
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .option("mergeSchema", "true")
            .partitionBy("symbol")  # Particiona por symbol para queries eficientes
            .saveAsTable(table_name)
        )
        
        print(f"✅ {df.count():,} registros salvos")
        
    except Exception as e:
        print(f"❌ Erro: {str(e)[:100]}")

print("\n✅ Todas as tabelas foram salvas com sucesso!")
print("\n📊 Tabelas criadas:")
print("  • default.crypto_ohlcv_15min  (15 minutos)")
print("  • default.crypto_ohlcv_1hour   (1 hora)")
print("  • default.crypto_ohlcv_4hour   (4 horas)")
print("\n🚀 Dados prontos para feature engineering!")

## Próximos Passos

Com os dados coletados de múltiplos pares e timeframes, agora podemos:

1. **Feature Engineering Multi-Timeframe:**
   - Agregar features de 15m, 1h e 4h para capturar diferentes escalas temporais
   - Criar features de correlação entre pares
   - Adicionar indicadores técnicos avançados

2. **Features de Sentiment (se APIs disponíveis):**
   - Funding rate
   - Open interest
   - Long/short ratio
   - Volume profile

3. **Modelagem Melhorada:**
   - Target adaptativo baseado em ATR
   - Balanceamento de classes mais sofisticado
   - Walk-forward validation

**Nota:** A coleta de 3 anos de dados para 8 pares em 3 timeframes resulta em ~100-150k candles, fornecendo uma base sólida para treinar modelos mais robustos.

In [0]:
((len(ohlcv)*60) / 60) / 24

**ccxt:** CryptoCurrency eXchange Trading - conectar e negociar em diversas exchanges de criptomoedas

**Alguns significados**
- O (Open / Abertura): O preço do ativo no exato momento em que o intervalo de tempo começou (ex: o preço às 10:00:00)
- H (High / Máxima): O preço mais alto que o ativo atingiu durante aquele intervalo
- L (Low / Mínima): O preço mais baixo que o ativo atingiu durante aquele intervalo
- C (Close / Fechamento): O preço do ativo no momento em que o intervalo terminou (ex: o preço às 10:59:59)
- V (Volume): A quantidade total do ativo que foi negociada (comprada/vendida) naquele período

In [0]:
import pyspark.sql.types as t
from pyspark.sql.window import Window
import pyspark.sql.functions as f

# criando dataframe
schema = t.StructType([
    t.StructField("timestamp"   ,t.LongType()     ,True)
    ,t.StructField("open"       ,t.DoubleType()   ,True)
    ,t.StructField("high"       ,t.DoubleType()   ,True)
    ,t.StructField("low"        ,t.DoubleType()   ,True)
    ,t.StructField("close"      ,t.DoubleType()   ,True)
    ,t.StructField("volume"     ,t.DoubleType()   ,True)
])

df_ohlcv        = spark.createDataFrame(ohlcv, schema)
df_ohlcv_btc    = spark.createDataFrame(btc_ohlcv, schema)

# convertendo a informação de data para timestamp
df_ohlcv        = df_ohlcv.withColumn("timestamp", (df_ohlcv.timestamp / 1000).cast("timestamp"))
df_ohlcv_btc    = df_ohlcv_btc.withColumn("timestamp", (df_ohlcv_btc.timestamp / 1000).cast("timestamp"))

 Não está no nosso fuso horário

In [0]:
from pyspark.sql.functions import from_utc_timestamp
df_ohlcv = df_ohlcv.withColumn("timestamp", from_utc_timestamp("timestamp", "America/Sao_Paulo"))
df_ohlcv_btc = df_ohlcv_btc.withColumn("timestamp", from_utc_timestamp("timestamp", "America/Sao_Paulo"))

df_ohlcv = df_ohlcv.orderBy(f.col('timestamp').desc())
df_ohlcv_btc = df_ohlcv_btc.orderBy(f.col('timestamp').desc())

In [0]:
# Renomeie as colunas do BTC para não dar conflito
df_ohlcv_btc_renamed = df_ohlcv_btc.select(
    f.col("timestamp"),
    f.col("close").alias("btc_close"),
    f.col("volume").alias("btc_volume")
)

# Join pela data/hora exata
df_ohlcv = df_ohlcv.join(df_ohlcv_btc_renamed, on="timestamp", how="left").orderBy("timestamp")

# Preenche eventuais nulos caso falte algum candle específico do BTC
df_ohlcv = df_ohlcv.fillna("ffill")

## Feature engineering

In [0]:
w = Window.orderBy("timestamp")

#### Log return (retorno logarítmico)
$$r_t = \ln(P_t) - \ln(P_{t-1})$$
- Cálculo do retorno de um ativo entre dois períodos usando logaritmos naturais
- Permite somar retornos de diferentes períodos
- adequado para séries temporais

In [0]:
df_ohlcv.display()

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

No dia 26 de março às 14:45, houve um pico de volume bem grande, porém, não mexeu nos preços seguintes. seria um sinal de exaustão?

In [0]:
df_feat = df_ohlcv.withColumn(
    "log_return",
    f.log(f.try_divide(f.col("close"), f.lag(f.col("close"), 1).over(w)))
)

df_feat = df_feat.withColumn(
    "btc_log_return",
    f.log(f.try_divide(f.col("btc_close"), f.lag(f.col("btc_close"), 1).over(w)))
)

# Retorno do par utilizado menos retorno do BTC (quem está performando melhor?)
df_feat = df_feat.withColumn("forca_relativa_btc", f.col("log_return") - f.col("btc_log_return"))
# df_feat.limit(5).display()

#### RSI (Índice de Força Relativa)
$$RSI = 100 - \left( \frac{100}{1 + RS} \right)$$
Onde o RS (Força Relativa) é:
$$RS = \frac{\text{media dos ganhos}}{\text{media das perdas}}$$

- Mede a velocidade e a mudança dos movimentos de preço
- Indica condições de sobrecompra (>70) ou sobrevenda (<30)
- Utilizado para identificar possíveis reversões de tendência
- Geralmente, utiliza-se um período de 14 candles para esse cálculo

In [0]:
periodo = 14

delta = f.col("close") - f.lag(f.col("close"), 1).over(w)

ganho = f.when(delta > 0, delta).otherwise(0)
perda = f.when(delta < 0, -delta).otherwise(0)

avg_ganho = f.avg(ganho).over(Window.orderBy("timestamp").rowsBetween(-periodo, 0))
avg_perda = f.avg(perda).over(Window.orderBy("timestamp").rowsBetween(-periodo, 0))

rs = avg_ganho / avg_perda
rsi = f.when(avg_perda == 0, 100).otherwise(100 - (100 / (1 + rs)))

df_feat = df_feat.withColumn("rsi", rsi)
df_feat.limit(5).display()


### Variação de volume
- mede a alteração na intensidade das negociações entre dois períodos
- valida se um movimento de preço tem relevância ou se é apenas um ruído de baixa liquidez
$$V_{change} = \frac{V_t - V_{t-1}}{V_{t-1}}$$
- Vt: Volume do período atual
- Vt-1: Volume do período anterior

In [0]:
df_feat = df_feat.withColumn(
    "vol_change"
    ,f.try_divide((f.col("volume") - f.lag(f.col("volume"), 1).over(w)), f.lag(f.col("volume"), 1).over(w))
)
# df_feat.limit(5).display()


### Médias móveis, desvio padrão móvel e z-score (janela de 20 períodos)

- **Média móvel:** Suaviza as flutuações de preço, mostrando a tendência geral
$$MA_{20} = \frac{1}{20} \sum_{i=0}^{19} P_{t-i}$$

- **Desvio padrão móvel:** Mede a volatilidade dos preços em uma janela de 20 períodos
$$STD_{20} = \sqrt{\frac{1}{20} \sum_{i=0}^{19} (P_{t-i} - MA_{20})^2}$$

- **Z-score:** Indica o quão distante o preço está da média móvel, em termos de desvio padrão
$$Z_{t} = \frac{P_t - MA_{20}}{STD_{20}}$$

- Pt: Preço do período atual
- MA20: Média móvel dos últimos 20 períodos
- STD20: Desvio padrão dos últimos 20 períodos

In [0]:
periodo = 20
w_movel = Window.orderBy("timestamp").rowsBetween(-periodo + 1, 0)

# Média móvel e Desvio padrão móvel do volume
df_feat = (
    df_feat
    # volume
    .withColumn("media_movel_vol", f.avg("volume").over(w_movel))
    .withColumn("media_movel_close", f.avg("close").over(w_movel))
    # desvio padrão
    .withColumn("std_movel_vol", f.stddev("volume").over(w_movel))
    .withColumn("std_movel_close", f.stddev("close").over(w_movel))
)
# df_feat.limit(5).display()

In [0]:
df_feat = (
    df_feat
    # volume
    .withColumn("z_score_vol", f.try_divide(f.col("volume") - f.col("media_movel_vol"), f.col("std_movel_vol")))
    # fechamento
    .withColumn("z_score_close", f.try_divide(f.col("close") - f.col("media_movel_close"), f.col("std_movel_close")))
)
# df_feat.limit(5).display()

In [0]:
# limite de anomalia (Z-Score > 3)
df_feat = df_feat.withColumn("vol_anomalia", f.col("z_score_vol") > 3)

# sinal que combina preço + anomalia de volume
df_feat = df_feat.withColumn(
    "vol_signal",
    f.when(
        (f.col("vol_anomalia")) & (f.col("close") > f.col("open")),
        "Forte compra (baleia)"
    ).when(
        (f.col("vol_anomalia")) & (f.col("close") < f.col("open")),
        "Forte venda (dump)"
    ).otherwise("Normal")
)
# df_feat.limit(5).display()

### Assimetria e Curtose (janela móvel)
- **Assimetria:** Mede a simetria da distribuição dos preços em relação à média. Valores positivos indicam cauda à direita, negativos à esquerda.
- **Curtose:** Mede o grau de "achatamento" da distribuição dos preços. Curtose alta indica mais extremos (caudas pesadas).
$$\text{Assimetria}_t = \text{skewness}(P_{t-19}, ..., P_t)$$
$$\text{Curtose}_t = \text{kurtosis}(P_{t-19}, ..., P_t)$$
- Calculadas sobre a janela móvel de 20 períodos usando a coluna de fechamento (`close`)

In [0]:
# assimetria e curtose
df_feat = df_feat.withColumn("assimetria", f.skewness("close").over(w_movel))
df_feat = df_feat.withColumn("curtosi", f.kurtosis("close").over(w_movel))
# df_feat.limit(5).display()

### Preço Típico
- preço médio ponderado de um candle
- utilizado em indicadores como o VWAP e alguns osciladores
$$\text{Typical Price}_t = \frac{H_t + L_t + C_t}{3}$$
- Ht: Máxima do período
- Lt: Mínima do período
- Ct: Fechamento do período

### VP
- VP: Valor negociado ponderado pelo preço típico
- Calculado como: VP = Typical_Price * volume

In [0]:
df_feat = (
  df_feat
  # Typical price
  .withColumn("typical_price", (f.col("high") + f.col("low") + f.col("close")) / 3)
  # VP
  .withColumn("VP", f.col("Typical_Price") * f.col("volume"))
)
# df_feat.limit(5).display()

### VWAP (Volume Weighted Average Price)
- preço médio ponderado pelo volume negociado em um período
- utilizado para avaliar o preço médio de execução de grandes ordens
$$\text{VWAP}_\text{daily} = \frac{\sum_{i=1}^{N} (\text{Typical Price}_i \times \text{Volume}_i)}{\sum_{i=1}^{N} \text{Volume}_i}$$
- N: número de candles no dia
- Typical Price: preço típico do candle
- Volume: volume negociado no candle

### Distância ao VWAP
- Mede o quanto o preço de fechamento está distante do VWAP diário
$$\text{Dist VWAP}_t = \frac{C_t - \text{VWAP}_\text{daily}}{\text{VWAP}_\text{daily}}$$
- Ct: Fechamento do período
- VWAP_daily: VWAP do dia

In [0]:
w_daily = Window.partitionBy(f.to_date("timestamp")).orderBy("timestamp").rowsBetween(Window.unboundedPreceding, 0)

# Daily VWAP
df_feat =(
  df_feat
  .withColumn("date", f.to_date("timestamp"))
  .withColumn("cum_VP", f.sum("VP").over(w_daily))
  .withColumn("cum_Volume", f.sum("volume").over(w_daily))
  .withColumn("daily_VWAP", f.try_divide(f.col("cum_VP"), f.col("cum_Volume")))
)

# Distância ao VWAP
df_feat = df_feat.withColumn("dist_VWAP", f.try_divide(f.col("close") - f.col("daily_VWAP"), f.col("daily_VWAP")))

# drop columns
df_feat = df_feat.drop("Typical_Price", "VP", "date", "cum_VP", "cum_Volume")
# df_feat.limit(10).display()

In [0]:
df_feat.groupBy('vol_signal').count().display()

In [0]:
(
    df_feat
    # .select(
    #     'timestamp'
    #     ,'open'
    #     ,'high'
    #     ,'low'
    #     ,'close'
    #     ,'volume'
    # )
    .write
    .format("delta")
    .mode("overwrite")
    .option('overwriteSchema', 'true')
    .saveAsTable("default.tcripto_features")
)